# Environment setup and Google Drive mount
In this section, we connect Google Drive to access the MVTec dataset and set up a secure destination to save the results. We also install any missing dependencies.

In [1]:
from google.colab import drive
import os
import shutil
import glob

# Mount Google Drive
drive.mount('/content/drive')

!git clone https://github.com/emanuelepietrocometti/sk-rd4ad.git
%cd /content/sk-rd4ad

!pip install --extra-index-url https://download.pytorch.org/whl/cu130 \
    torch>=2.1.0 torchvision>=0.16.0 numpy pandas scipy imageio matplotlib \
    opencv-python opencv-contrib-python Pillow scikit-image scikit-learn \
    fastprogress geomloss tqdm

Mounted at /content/drive
Cloning into 'sk-rd4ad'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 211 (delta 16), reused 11 (delta 5), pack-reused 180 (from 1)
Receiving objects: 100% (211/211), 495.56 KiB | 23.60 MiB/s, done.
Resolving deltas: 100% (117/117), done.
/content/sk-rd4ad


# Configuration variables (CURRENT RUN)
Set the dataset class, the run name and the Drive base folder.
Every run gets its own folder on Drive with the following layout:

```
<DRIVE_BASE_PATH>/<RUN_NAME>/
    checkpoints/     # .pth weights
    results/         # heatmaps, metric reports, eval output
    run_config.json  # snapshot of the hyperparameters used
```

Update `RUN_NAME` for each new iteration so nothing gets overwritten.

In [2]:
import os
import json
from datetime import datetime

# === CONFIGURE YOUR RUN HERE ===
CLASS_NAME = "carpet"
RUN_NAME   = "23072026_run_01_carpet"

# Base folder on Drive: every run creates a sub-folder here
DRIVE_BASE_PATH = "/content/drive/MyDrive/Tesi/results_SKRD4AD"

# Dataset location on Drive
DRIVE_DATASET_PATH = f"/content/drive/MyDrive/Tesi/MVTec/{CLASS_NAME}"

# --- Derived Drive paths (one folder per run) ---
DRIVE_RUN_PATH         = os.path.join(DRIVE_BASE_PATH, RUN_NAME)
DRIVE_CHECKPOINTS_PATH = os.path.join(DRIVE_RUN_PATH, "checkpoints")
DRIVE_RESULTS_PATH     = os.path.join(DRIVE_RUN_PATH, "results")

# --- Local paths (fast Colab storage, used during training) ---
LOCAL_DATASET_PATH     = f"./mvtec/{CLASS_NAME}"
LOCAL_CHECKPOINTS_PATH = "./checkpoints"
LOCAL_RESULTS_PATH     = "./results"

# Prefix used by main.py to name the .pth files
PROJECT_NAME = f"skrd4ad_{RUN_NAME}"

# --- Hyperparameters for this run (single source of truth) ---
HPARAMS = {
    "seed": 42,
    "L2": 2,
    "res": 3,
    "rate": 0.5,
    "batch_size": 16,
    "layerloss": 1,
    "seg": 1,
    "vis": 0,
    "print_epoch": 50,
    "learning_rate": 0.005,
    "epochs": 200,
    "cut": 1,
}

print("Drive run folder :", DRIVE_RUN_PATH)
print("  checkpoints ->", DRIVE_CHECKPOINTS_PATH)
print("  results     ->", DRIVE_RESULTS_PATH)

Drive run folder : /content/drive/MyDrive/Tesi/results_SKRD4AD/23072026_run_01_carpet
  checkpoints -> /content/drive/MyDrive/Tesi/results_SKRD4AD/23072026_run_01_carpet/checkpoints
  results     -> /content/drive/MyDrive/Tesi/results_SKRD4AD/23072026_run_01_carpet/results


# Local dataset preparation
Deep Learning frameworks are significantly slower when reading images directly from Google Drive. This cell copies the dataset to Colab's local storage and creates both the local output folders and the destination folders on Drive.

In [3]:
import shutil

# --- Guard: refuse to silently overwrite an existing run on Drive ---
if os.path.exists(DRIVE_RUN_PATH) and os.listdir(DRIVE_RUN_PATH):
    raise RuntimeError(
        f"Run folder already exists and is not empty: {DRIVE_RUN_PATH}\n"
        "Change RUN_NAME, or delete the folder manually if you want to overwrite it."
    )

print(f"Copying the {CLASS_NAME} dataset from Drive to local Colab storage...")

if os.path.exists(LOCAL_DATASET_PATH):
    shutil.rmtree(LOCAL_DATASET_PATH)
shutil.copytree(DRIVE_DATASET_PATH, LOCAL_DATASET_PATH)
print("Dataset successfully copied!")

# Start each run from clean local output folders
for path in (LOCAL_CHECKPOINTS_PATH, LOCAL_RESULTS_PATH):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

# Create the run layout on Drive
os.makedirs(DRIVE_CHECKPOINTS_PATH, exist_ok=True)
os.makedirs(DRIVE_RESULTS_PATH, exist_ok=True)

# Snapshot the configuration next to the results
with open(os.path.join(DRIVE_RUN_PATH, "run_config.json"), "w") as f:
    json.dump(
        {
            "run_name": RUN_NAME,
            "class_": CLASS_NAME,
            "project_name": PROJECT_NAME,
            "dataset": DRIVE_DATASET_PATH,
            "started_at": datetime.now().isoformat(timespec="seconds"),
            "hparams": HPARAMS,
        },
        f,
        indent=2,
    )

print("Run layout ready on Drive.")

Copying the carpet dataset from Drive to local Colab storage...
Dataset successfully copied!
Run layout ready on Drive.


# Training phase
Execute `main.py` using the local (fast) storage. Weights land in `./checkpoints`, heatmaps and metric reports in `./results`; both are synced to Drive at the end of the notebook.

In [4]:
print(f"--- STARTING TRAINING FOR CLASS: {CLASS_NAME} ---")

!python main.py \
  --class_ {CLASS_NAME} \
  --data_path ./mvtec/ \
  --ckpt_path {LOCAL_CHECKPOINTS_PATH}/ \
  --img_path {LOCAL_RESULTS_PATH}/ \
  --project_name {PROJECT_NAME} \
  --seed {HPARAMS["seed"]} \
  --L2 {HPARAMS["L2"]} \
  --res {HPARAMS["res"]} \
  --rate {HPARAMS["rate"]} \
  --batch_size {HPARAMS["batch_size"]} \
  --layerloss {HPARAMS["layerloss"]} \
  --seg {HPARAMS["seg"]} \
  --vis {HPARAMS["vis"]} \
  --print_epoch {HPARAMS["print_epoch"]} \
  --learning_rate {HPARAMS["learning_rate"]} \
  --epochs {HPARAMS["epochs"]} \
  --cut {HPARAMS["cut"]}

--- STARTING TRAINING FOR CLASS: carpet ---
--------args----------
epochs: 200
res: 3
learning_rate: 0.005
batch_size: 16
seed: [42]
class_: carpet
seg: 1
print_epoch: 50
data_path: ./mvtec/
ckpt_path: ./checkpoints/
project_name: skrd4ad_23072026_run_01_carpet
print_canshu: 1
score_num: 1
print_loss: 1
img_path: ./results/
vis: 0
cut: 1
layerloss: 1
rate: 0.5
print_max: 1
net: wide_res50
L2: 2
--------args----------

*************************
seed: 42
cuda
carpet
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-95faca4d.pth
100% 132M/132M [00:08<00:00, 16.8MB/s]
epoch [1/200], loss:2.5341
epoch [2/200], loss:0.7327
epoch [3/200], loss:0.5576
epoch [4/200], loss:0.4669
epoch [5/200], loss:0.3829
epoch [6/200], loss:0.3519
epoch [7/200], loss:0.3397
epoch [8/200], loss:0.3798
epoch [9/200], loss:0.3271
epoch [10/200], loss:0.3006
epoch [11/200], loss:0.2835
epoch [12/200], loss:0.2726
epoch [13/200], lo

# Evaluation and metrics report
Run `eval.py` to generate the final report (AUPRO, AP-loc, F1-Score, ...) and the heatmaps grouped by TP / TN / FP / FN. The script picks up the most recent checkpoint produced by the training cell.

In [5]:
import glob

print(f"--- STARTING EVALUATION FOR CLASS: {CLASS_NAME} ---")

checkpoints = glob.glob(os.path.join(LOCAL_CHECKPOINTS_PATH, "*.pth"))

if not checkpoints:
    raise FileNotFoundError(
        "No checkpoints found. The training phase might have failed."
    )

# Newest by modification time
LATEST_CHECKPOINT = max(checkpoints, key=os.path.getmtime)
print(f"Using checkpoint: {LATEST_CHECKPOINT}")

EVAL_IMG_PATH = os.path.join(LOCAL_RESULTS_PATH, "eval_report")
os.makedirs(EVAL_IMG_PATH, exist_ok=True)

!python eval.py \
  --class_ {CLASS_NAME} \
  --data_path ./mvtec/ \
  --checkpoint_path {LATEST_CHECKPOINT} \
  --img_path {EVAL_IMG_PATH}/

--- STARTING EVALUATION FOR CLASS: carpet ---
Using checkpoint: ./checkpoints/skrd4ad_23072026_run_01_carpet_ep100_seed42_sample_auc=1.0000.pth
Using device: cuda
Loading checkpoint from: ./checkpoints/skrd4ad_23072026_run_01_carpet_ep100_seed42_sample_auc=1.0000.pth
Phase 1/2: Extracting anomaly scores and spatial maps...
Calculating pixel-level metrics (AUPRO, AP-loc, AUROC). This may take a moment...
 EVALUATION METRICS REPORT 
--- SAMPLE LEVEL (Image Classification) ---
Optimal Threshold: 0.6250
AUROC:             0.9996
Accuracy:          0.9915
F1-Score:          0.9944
Precision:         0.9889
Recall:            1.0000

--- PIXEL LEVEL (Defect Localization) ---
AUPRO:             0.9762
AP-loc:            0.6743
Pixel AUROC:       0.9918

Report saved to: ./results/eval_report/report.txt

Phase 2/2: Saving visualizations categorized by Confusion Matrix...
Done! Evaluated images sorted in ./results/eval_report/
Confusion matrix saved to ./results/eval_report/confusion_matrix.png

# Hyperparameter fine-tuning (optional)
Run this block only if you want to execute the hyperparameter search. Its output is written inside the run's `results/fine_tuning` folder.

In [ ]:
print(f"--- STARTING HYPERPARAMETER TUNING FOR CLASS: {CLASS_NAME} ---")

LOCAL_FINETUNE_PATH = os.path.join(LOCAL_RESULTS_PATH, "fine_tuning")
os.makedirs(LOCAL_FINETUNE_PATH, exist_ok=True)

!python hyperparameters_fine_tuning.py \
  --class_ {CLASS_NAME} \
  --data_path ./mvtec/ \
  --save_path {LOCAL_FINETUNE_PATH}/

# Save results to Google Drive
Final synchronization: weights go to `<RUN_NAME>/checkpoints`, everything else (heatmaps, metric reports, eval output, fine-tuning) to `<RUN_NAME>/results`.

In [6]:
def sync_to_drive(local_dir, drive_dir):
    """Copy a local folder to Drive, creating the destination if needed."""
    if not os.path.isdir(local_dir):
        print(f"  [skip] {local_dir} does not exist")
        return 0
    os.makedirs(drive_dir, exist_ok=True)
    shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
    n = sum(len(files) for _, _, files in os.walk(drive_dir))
    print(f"  [ok]   {local_dir} -> {drive_dir}  ({n} files)")
    return n


print(f"Syncing run '{RUN_NAME}' to Drive...")

sync_to_drive(LOCAL_CHECKPOINTS_PATH, DRIVE_CHECKPOINTS_PATH)
sync_to_drive(LOCAL_RESULTS_PATH, DRIVE_RESULTS_PATH)

# Record when the run finished
config_file = os.path.join(DRIVE_RUN_PATH, "run_config.json")
with open(config_file) as f:
    cfg = json.load(f)
cfg["finished_at"] = datetime.now().isoformat(timespec="seconds")
with open(config_file, "w") as f:
    json.dump(cfg, f, indent=2)

print(f"\nTRANSFER COMPLETE. Everything is saved in: {DRIVE_RUN_PATH}")

Syncing run '23072026_run_01_carpet' to Drive...
  [ok]   ./checkpoints -> /content/drive/MyDrive/Tesi/results_SKRD4AD/23072026_run_01_carpet/checkpoints  (2 files)
  [ok]   ./results -> /content/drive/MyDrive/Tesi/results_SKRD4AD/23072026_run_01_carpet/results  (119 files)

TRANSFER COMPLETE. Everything is saved in: /content/drive/MyDrive/Tesi/results_SKRD4AD/23072026_run_01_carpet


In [ ]:
!pip install onnx onnxruntime-gpu==1.22.0

# ONNX export (optional)
Exports the best checkpoint of this run. The `.onnx` file is written into the run's `results/exports` folder on Drive.

In [ ]:
EXPORT_PATH = os.path.join(LOCAL_RESULTS_PATH, "exports")
os.makedirs(EXPORT_PATH, exist_ok=True)

# Reuse the checkpoint selected in the evaluation cell, or pick the newest one
try:
    ckpt = LATEST_CHECKPOINT
except NameError:
    ckpt = max(glob.glob(os.path.join(LOCAL_CHECKPOINTS_PATH, "*.pth")),
               key=os.path.getmtime)

print(f"Exporting: {ckpt}")
!python export_onnx_from_checkpoint.py "{ckpt}" "{EXPORT_PATH}"

sync_to_drive(EXPORT_PATH, os.path.join(DRIVE_RESULTS_PATH, "exports"))